In [18]:
from langchain.chat_models import init_chat_model
from typing import Annotated,List,Dict,Literal,Optional,Any
import operator
from langchain.agents import  create_agent
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace
from langchain_community.tools import DuckDuckGoSearchResults
from langchain.agents.middleware import ToolCallLimitMiddleware,ModelRetryMiddleware,HumanInTheLoopMiddleware
import re
from langchain.tools import tool
from langchain.messages import HumanMessage
from langchain_mistralai import  ChatMistralAI
from pydantic import BaseModel,Field
import os
from dotenv import load_dotenv
from langgraph_supervisor import create_supervisor

load_dotenv()

True

In [23]:
model_MIS=ChatMistralAI()

In [20]:
model.invoke('hi how are u')

AIMessage(content="Hi! 😊 I'm just a computer program, so I don't have feelings, but I'm here and ready to help you with anything you need! How about you—how are you doing today? 🌟", additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 7, 'total_tokens': 54, 'completion_tokens': 47, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small', 'model': 'mistral-small', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019c7afb-7452-7123-a3df-60fbf7354156-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 47, 'total_tokens': 54})

In [21]:
tool=DuckDuckGoSearchResults(top=100,name='duckduckgo')

In [8]:

llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-R1-0528",
    task="text-generation",
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.03,
    provider="auto", 
    async_client=True,
    huggingfacehub_api_token=os.getenv('HF_TOKEN') ,# let Hugging Face choose the best provider for you,

)


model=ChatHuggingFace(llm=llm)

/Users/divyyadav/researcher-agent/researcher-agent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [40]:
from langchain.tools import tool
from langchain_core.messages import AIMessage
from langgraph.checkpoint.memory import InMemorySaver 
from langgraph.types import Command

def filter_response(response: AIMessage) -> str:
    """
    Filters out the reasoning tags from the response of the model.
    """
    content = response.content
    cleaned_content = re.sub(r'<think>.*?</think>\n*', '', content, flags=re.DOTALL).strip()
    return cleaned_content

In [10]:
agent = create_agent(
    model=model,
    tools=[tool],
    name='Researcher-Agent',
    system_prompt='You are a research assistant. Use the search tool to find information and provide detailed analysis.',
    middleware=[ModelRetryMiddleware(max_retries=3)]
      )



In [60]:
@tool(description="Sends an email to the specified recipient with the given content")
def send_email(input:str)->str:
    """Sends an email to the specified recipient with the given content with the given input"""
    output=model.invoke(f'make an input using the given input to send an email {input}')
    return output.content

In [64]:
agent=create_agent(
    model=model_MIS,
    tools=[send_email],
    name='Email-Agent',
    system_prompt='You are as aassistant whose job is to send emails from Divy',
    middleware=[HumanInTheLoopMiddleware(interrupt_on={'send_email':  True})],
    checkpointer=InMemorySaver())

In [62]:
config={'configurable':{'thread_id': '12345'}}

In [57]:
a=model.invoke('hi how are u')

In [59]:
a.content

"Hi! I'm just a bunch of code, so I don't have feelings, but I'm here and ready to help you with anything you need! 😊 How about you? How are you doing today?"

In [63]:
output=await agent.ainvoke({'messages': [HumanMessage("Send an email to Divy about the meeting schedule for tomorrow.")]},config=config)

In [65]:
output['__interrupt__']

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'input': 'Subject: Meeting Schedule for Tomorrow\n\nHi Divy,\n\nI hope this email finds you well. I am writing to inform you about the meeting schedule for tomorrow.\n\nWe have the following meetings planned:\n\n1. Team Sync-Up: 10:00 AM - 11:00 AM\n2. Project Review with Client: 1:00 PM - 2:30 PM\n3. Departmental Meeting: 3:00 PM - 4:30 PM\n\nPlease ensure you are prepared for these meetings and join on time. If you have any conflicts or need to reschedule, please let me know as soon as possible.\n\nLooking forward to a productive day tomorrow!\n\nBest regards,\n[Your Name]'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'input': 'Subject: Meeting Schedule for Tomorrow\\n\\nHi Divy,\\n\\nI hope this email finds you well. I am writing to inform you about the meeting schedule for tomorrow.\\n\\nWe have the following meetings planned:\\n\\n1. Team Sync-Up: 10:00 AM - 11:00 AM\\n2. Projec

In [66]:
agent.invoke(Command(resume=
    {"decisions": [{"type": "edit", "edited_action": {
        "name": "send_email",
        "args": {"input": "the meeting must be at 2am"}
    }}]}),
    config=config
)

{'messages': [AIMessage(content="Sure, I can help you draft an email. Could you please provide me with the following details:\n\n1. **Recipient's Email Address**: Who is the email for?\n2. **Subject Line**: What is the subject of the email?\n3. **Main Content**: What would you like the email to say?\n4. **Tone**: Should the email be formal, casual, or friendly?\n5. **Any Attachments**: Do you need to attach any files?\n\nOnce I have these details, I can help you draft the perfect email!", additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 18, 'total_tokens': 131, 'completion_tokens': 113, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small', 'model': 'mistral-small', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, name='Email-Agent', id='lc_run--019c7b13-ab48-7601-9d12-deb87af884fb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 113, 'total_tokens': 131})]}